# Open Notebook in Colab 

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/discovery_tutorial.ipynb)


Please cite this if you use llm_tracker: <br>
Low, D.M., Mair, P., Nock, M., & Ghosh, S. (2026). Text Psychometrics: Assessing Psychological Constructs in Text Using Natural Language Processing. PLOS Digital Health.

# Discover constructs and build a codebook

This tutorial starts with documents and a theoretical framework, rather than a
predefined codebook. It discovers constructs, consolidates their names, lets you
review and edit the groups, and builds a codebook for the coding pipeline.

**Requires the updated discovery module supplied with the discovery refactor.**
If you replaced the module while a notebook was open, restart the kernel first.
Use the Python environment where `llm_tracker` and Jupyter are installed.

Discovery and semantic merging make LLM calls. Manual edits, filtering, formatting,
and codebook creation run locally. An optional reevaluation step makes another
LLM call. No API key is stored in this notebook.


# Setup (first time only)

You only need this notebook and an internet connection. Run the next cell to
download LLM Tracker and the practice files, install the package and
TextDescriptives, and download the English language model used below. You do
not need to clone or download the repository beforehand.

This works in Google Colab or a local Jupyter notebook using Python 3.11–3.13.
The first setup may take a few minutes. If asked to restart the session/kernel,
restart it and rerun this setup cell before continuing. Setup does not call an
LLM or spend API credits.

In [1]:
from pathlib import Path
import subprocess
import sys
import tempfile
from urllib.request import urlretrieve
from zipfile import ZipFile

# Find an existing copy, or download LLM Tracker and the sample data.
project_dir = Path.cwd()
if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = project_dir.parent

if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = Path.cwd() / "llm_tracker-main"
    if not (project_dir / "src" / "llm_tracker").is_dir():
        print("Downloading LLM Tracker...")
        with tempfile.TemporaryDirectory() as folder:
            archive = Path(folder) / "llm_tracker.zip"
            urlretrieve(
                "https://codeload.github.com/childmindresearch/llm_tracker/zip/refs/heads/main",
                archive,
            )
            with ZipFile(archive) as downloaded:
                downloaded.extractall(project_dir.parent)

# Allow NumPy 2 for Python 3.13 compatibility.
pyproject = project_dir / "pyproject.toml"
settings = pyproject.read_text()
settings = settings.replace('numpy = "^1.26"', 'numpy = ">=1.26,<3"')
pyproject.write_text(settings)

# Install the package; stop if installation fails.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--only-binary=numpy,pandas,scikit-learn",
        "--timeout", "30", "--retries", "1",
        "-e", str(project_dir),
    ],
    check=True,
)

sys.path.insert(0, str(project_dir / "src"))

# Make the bundled sample-data paths work.
%cd "{project_dir}"

import llm_tracker.discovery_v2
print("Discovery setup complete.")

Obtaining file:///Users/freymon.perez/Projects/GitHub/llm_tracker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llm-tracker (pyproject.toml): started
  Building editable for llm-tracker (pyproject.toml): finished with status 'done'
  Created wheel for llm-tracker: filename=llm_tracker-0.1.0b1-py3-none-any.whl size=5103 sha256=d9e045ceea284e9c01ec2c0fe6d9e46d90a9c7446ebf8c579a59479d28325c33
  Stored in directory: /private/var/folders/09/drdv5jw504142vpxfq711jf815q1qc/T/pip-ephem-wheel-cache-b61hb6eh/wheels/9d/32/98/c6659a


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
/Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/models.py:40: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


Discovery setup complete.


## 1. Set paths and options

The paths below start with the repository's Reddit sample. For your own data, keep
`use_demo_data=False` and change `input_path`. For a small synthetic example, set
`use_demo_data=True`; the next setup cell then uses `data/discovery_demo.csv`.
Relative paths are resolved from the notebook's working directory.

- TXT: one document per file; set `text_column=None` and `id_column=None`.
- CSV/XLSX/XLS: set `text_column` for one document per row; `id_column` is optional.
- A directory is read nonrecursively. The column settings apply to every tabular file.

For tabular data, omitting `text_column` treats the entire table as one document.
Set `model_name` to an available model ID for your provider; leaving it `None`
uses the package's configured default. Set `OPENROUTER_API_KEY` in your environment
or specify a local `.env` file. Never commit that file or print your key.


In [ ]:
from pathlib import Path

use_demo_data = True

framework = "Cognitive behavioral therapy"  # Modern clinical psychology constructs (CBT, DBT, ACT, etc.)

input_path = Path("sample_data/reddit_autism_anxiety_depression.csv")
output_root = Path("discovery_outputs")

text_column = "post"
id_column = None  # Uses the row number as each document's ID.

provider = "openrouter"
model_name = "google/gemini-3.1-pro-preview" # Or your chosen provider model ID.
env_file = "" # Or Path("/absolute/path/to/.env")
max_retries = 2

fuzzy_threshold = 0.9  # None disables fuzzy spelling matching.
seed = 0
run_reevaluation = False  # True makes an additional LLM call after filtering.
n_codebook_examples = 3


In [7]:
import json
from copy import deepcopy

import pandas as pd
from IPython.display import display

import llm_tracker.discovery_v2 as discovery_module
from llm_tracker.config import AnalyzerConfig
from llm_tracker.discovery_v2 import (
    DISCOVERY_PROMPT,
    MERGE_PROMPT,
    REEVALUATE_PROMPT,
    LLMTrackerDiscoverer,
    apply_merge_edits,
    build_codebook,
    build_discovery_tables,
    codebook_to_frame,
    collapse_variants,
    filter_merged_constructs,
    format_merged_constructs,
    frame_to_codebook,
    merge_constructs,
    reevaluate_merged_constructs,
    save_codebook,
)

print("Discovery module:", discovery_module.__file__)
print("Working directory:", Path.cwd())


Discovery module: /Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/discovery_v2.py
Working directory: /Users/freymon.perez/Projects/GitHub/llm_tracker


The demo cell creates a CSV only when `use_demo_data=True`. Change the input path
before using your own data. An existing demo file is left untouched.


In [8]:
if use_demo_data:
    input_path = Path("data/discovery_demo.csv")
    text_column, id_column = "text", "doc_id"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        demo = pd.DataFrame([
            {"doc_id": "demo_1", "text": "If I make one mistake at work, everyone will think I am incompetent."},
            {"doc_id": "demo_2", "text": "I assume my friend is angry whenever she takes a while to reply."},
            {"doc_id": "demo_3", "text": "I felt anxious about the meeting, so I avoided opening the invitation."},
            {"doc_id": "demo_4", "text": "I wrote down the evidence for and against my fear, which helped me consider another explanation."},
            {"doc_id": "demo_5", "text": "I took one small step toward the task instead of putting it off again."},
            {"doc_id": "demo_6", "text": "The delivery contained three blue folders and a box of paper clips."},
        ])
        demo.to_csv(input_path, index=False)
    display(pd.read_csv(input_path))

if not input_path.exists():
    raise FileNotFoundError(f"Update input_path; no input exists at {input_path.resolve()}")

discovery_dir = output_root / "discovery"
merge_dir = output_root / "merged"
edits_dir = output_root / "edited"
filtered_dir = output_root / "filtered"
reevaluation_dir = output_root / "reevaluated"
tables_dir = output_root / "final_tables"
codebook_dir = output_root / "codebooks"
output_root.mkdir(parents=True, exist_ok=True)


,doc_id,text
0,demo_1,"If I make one mistake at work, everyone will t..."
1,demo_2,I assume my friend is angry whenever she takes...
2,demo_3,"I felt anxious about the meeting, so I avoided..."
3,demo_4,I wrote down the evidence for and against my f...
4,demo_5,I took one small step toward the task instead ...
5,demo_6,The delivery contained three blue folders and ...


## 2. Configure the discoverer and inspect its prompt

The discovery prompt uses `{framework}` and `{document}`. Keep these placeholders
when customizing it. Literal JSON braces in a template must be doubled: `{{` and `}}`.
Printing a template does not call the API.


In [9]:
config_kwargs = {
    "provider": provider,
    "max_retries": max_retries,
    "temperature": 0.0,
}
if model_name is not None:
    config_kwargs["model_name"] = model_name
if env_file is not None:
    config_kwargs["api_key"] = str(env_file)

config = AnalyzerConfig(**config_kwargs)
discoverer = LLMTrackerDiscoverer(config=config)
print("Model:", config.model_name)
print(discoverer.discovery_prompt)


Model: google/gemini-3.1-pro-preview
You are an expert qualitative researcher analyzing a document through the lens of a specific theoretical framework.

Framework: {framework}

Read the document below and identify the constructs from this framework that are present in the text. Use your own judgement about what counts as a construct of this framework; you are not working from a fixed list.

For each construct you find, provide:
- "construct": a short name for the construct
- "description": one sentence describing the construct as it appears here
- "quotes": a list of verbatim quotes from the document that evidence it

Only report constructs genuinely grounded in the text. If none are present, return an empty list.
Spell out construct names and descriptions in full. Do not use acronyms such as
CBT; write cognitive behavioral therapy instead.

Respond ONLY with a JSON object of the form:
{{"constructs": [{{"construct": "...", "description": "...", "quotes": ["..."]}}]}}

Document:
{docu

In [10]:
# Replace the prompt with a simple message so we can see that it changed.
discoverer.discovery_prompt = "We changed the prompt completely!"
print(discoverer.discovery_prompt)

# Change it back to the default prompt before running discovery.
discoverer.discovery_prompt = DISCOVERY_PROMPT
assert discoverer.discovery_prompt == DISCOVERY_PROMPT


We changed the prompt completely!


## 3. Discover constructs in the documents

This cell makes LLM calls: normally one per document, with retries as configured.
Rerunning it sends every input document again. Later steps use the dataframe in
memory and do not repeat discovery.

The result has one row per supporting quote. A detected construct with no quote
gets an empty-quote row. Successful documents with no detected constructs have no
rows, but still count toward the frequency denominator. Failed documents do not.
The tutorial stops on failures so you can review them before interpreting results.


In [11]:
discoveries, discovery_metadata, discovery_errors = discoverer.discover(
    path=input_path,
    framework=framework,
    text_column=text_column,
    id_column=id_column,
    output_dir=discovery_dir,
    save_format="both",
)

print("Successfully processed documents:", discoveries.attrs["total_documents"])
print("Discovery rows:", len(discoveries))
print("Failures:", len(discovery_errors))
display(discoveries)

if discovery_errors:
    display(pd.DataFrame([error.model_dump() for error in discovery_errors]))
    raise RuntimeError("Review the saved discovery errors before continuing. Rerunning discovery processes every document again.")
if discoveries.empty:
    raise RuntimeError("No constructs were discovered. Review the inputs, framework, and prompt before merging.")


Discovering across 6 document(s) from 1 file(s).

Discovering [1/6]: discovery_demo.csv::demo_1
  OK: 3 construct(s), 3 quote(s)
Discovering [2/6]: discovery_demo.csv::demo_2
  OK: 2 construct(s), 2 quote(s)
Discovering [3/6]: discovery_demo.csv::demo_3
  OK: 2 construct(s), 2 quote(s)
Discovering [4/6]: discovery_demo.csv::demo_4
  OK: 2 construct(s), 2 quote(s)
Discovering [5/6]: discovery_demo.csv::demo_5
  OK: 2 construct(s), 2 quote(s)
Discovering [6/6]: discovery_demo.csv::demo_6
  OK: 0 construct(s), 0 quote(s)

Discovery complete: 6/6 document(s), 11 unique construct name(s), 0 error(s).
Saved discovery output to discovery_outputs/discovery
Successfully processed documents: 6
Discovery rows: 11
Failures: 0


,filename,doc_id,construct,description,quote
0,discovery_demo.csv,demo_1,Cognitive Distortion: Mind Reading,The individual assumes they know the thoughts ...,everyone will think I am incompetent.
1,discovery_demo.csv,demo_1,Intermediate Belief: Conditional Assumption,"The individual operates under a rigid, dysfunc...","If I make one mistake at work, everyone will t..."
2,discovery_demo.csv,demo_1,Cognitive Distortion: All-or-Nothing Thinking,The individual views a single flaw or error as...,"If I make one mistake at work, everyone will t..."
3,discovery_demo.csv,demo_2,Cognitive distortion,The individual exhibits a pattern of biased th...,I assume my friend is angry whenever she takes...
4,discovery_demo.csv,demo_2,Automatic thought,"The text illustrates an immediate, reflexive n...",I assume my friend is angry whenever she takes...
5,discovery_demo.csv,demo_3,Avoidance Behavior,A behavioral response characterized by evading...,so I avoided opening the invitation.
6,discovery_demo.csv,demo_3,Emotional Response,The affective state or feeling of distress exp...,I felt anxious about the meeting
7,discovery_demo.csv,demo_4,Examining the evidence,The process of objectively evaluating the fact...,I wrote down the evidence for and against my fear
8,discovery_demo.csv,demo_4,Cognitive restructuring,"The process of developing new, more balanced o...",which helped me consider another explanation.
9,discovery_demo.csv,demo_5,Graded task assignment,The cognitive behavioral strategy of breaking ...,I took one small step toward the task


## 4. Inspect name normalization before semantic merging

This step protects against small variations in naming (e.g., cognitive behavioral therapy; cognitive behavior therapy) before being interpreted as entirely different constructs.

This step involves no LLM calls. Case, separators, and extra spaces are normalized; constituent
names become lowercase. Optional fuzzy matching will also match near spellsings, 
please note this is based on spelling only, not meaning. To avoid this simply set
`fuzzy_threshold=None` to use the most basic name normalization.

`count` in this intermediate table counts distinct `(filename, doc_id)` pairs.
`instance_count` counts nonempty quote occurrences before deduplication. The final
merge table calls the document count `document_count`.


In [12]:
normalized = collapse_variants(discoveries, fuzzy_threshold=fuzzy_threshold)
display(normalized[["construct", "count", "instance_count", "variants"]])
display(pd.DataFrame(normalized.attrs.get("name_matches", [])))


,construct,count,instance_count,variants
0,automatic thought,1,1,[Automatic thought]
1,avoidance behavior,1,1,[Avoidance Behavior]
2,cognitive distortion,1,1,[Cognitive distortion]
3,cognitive distortion all or nothing thinking,1,1,[Cognitive Distortion: All-or-Nothing Thinking]
4,cognitive distortion mind reading,1,1,[Cognitive Distortion: Mind Reading]
5,cognitive restructuring,1,1,[Cognitive restructuring]
6,emotional response,1,1,[Emotional Response]
7,examining the evidence,1,1,[Examining the evidence]
8,graded task assignment,1,1,[Graded task assignment]
9,intermediate belief conditional assumption,1,1,[Intermediate Belief: Conditional Assumption]


,original_name,construct_found
0,Automatic thought,automatic thought
1,Avoidance Behavior,avoidance behavior
2,Cognitive Distortion: All-or-Nothing Thinking,cognitive distortion all or nothing thinking
3,Cognitive Distortion: Mind Reading,cognitive distortion mind reading
4,Cognitive distortion,cognitive distortion
5,Cognitive restructuring,cognitive restructuring
6,Emotional Response,emotional response
7,Examining the evidence,examining the evidence
8,Graded task assignment,graded task assignment
9,Intermediate Belief: Conditional Assumption,intermediate belief conditional assumption


## 5. Merge constructs by meaning

The LLM now looks for discovered constructs that describe related ideas. It puts
them into groups and gives each group a name and definition. For example, it might
place "self hate" and "low self esteem" under "Negative self-perception".

You can print or edit the instructions below. Keep `{framework}`, `{constructs}`,
and `{existing_note}` in the prompt so the function can fill in the information.
The example restores the default merge prompt before running.

These are the main settings in the next call to `merge_constructs()`:

| Setting | What it means |
|---|---|
| `df` | The discovery results we want to organize. Keep the full original results. |
| `config` | The model and API settings already chosen above. |
| `framework` | The perspective the model uses to decide which ideas belong together. |
| `merge_prompt` | The written instructions for grouping the constructs. |
| `fuzzy_threshold` | How similar two spellings must be before treating them as one name. A higher number is stricter; `None` turns spelling matching off. |
| `seed` | Keeps the shuffle of the construct list repeatable when you reuse the same number. It does not guarantee identical LLM answers. |
| `batch` | `False` asks the model to consider the list in one request. `True` divides it into smaller, sequential requests. |
| `batch_size` | When batching, how many discovered construct names go into each request; the default is 150. |
| `max_prompt_chars` | A size check that stops an oversized single-pass merge before it is sent. |
| `existing` | Optional earlier grouping to show the model; this walkthrough starts a new grouping. |
| `total_documents` | Usually filled in automatically from discovery. It includes successful documents with no findings. Supply it yourself only when restoring results that lost this information. |
| `output_dir` | Where the results should be saved. |
| `save_format` | `"both"` saves a spreadsheet-friendly CSV and a JSON file that preserves lists. |


In [13]:
merged, merge_table = merge_constructs(
    df=discoveries,
    config=config,
    framework=framework,
    merge_prompt=MERGE_PROMPT,
    fuzzy_threshold=fuzzy_threshold,
    seed=seed,
    batch=False,
    max_prompt_chars=60000,
    output_dir=merge_dir,
    save_format="both",
)

display(format_merged_constructs(merge_table, n_quotes=3, seed=seed))


Collapsed 11 discovered name(s) into 11 distinct construct(s).
Saved merge output to discovery_outputs/merged


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Automatic Thought,"An immediate, reflexive negative cognition tha...",merged,automatic thought,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
1,Avoidance Behavior,A behavioral response characterized by evading...,merged,avoidance behavior,1,1,16.666667,6,1.00,so I avoided opening the invitation.,16.7% (1/6 documents)
2,Avoidance Behavior,A behavioral response characterized by evading...,merged,overcoming behavioral avoidance,1,1,16.666667,6,0.75,instead of putting it off again.,16.7% (1/6 documents)
3,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
4,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion all or nothing thinking,1,1,16.666667,6,0.90,"If I make one mistake at work, everyone will t...",16.7% (1/6 documents)
5,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion mind reading,1,1,16.666667,6,0.90,everyone will think I am incompetent.,16.7% (1/6 documents)
6,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,cognitive restructuring,1,1,16.666667,6,1.00,which helped me consider another explanation.,16.7% (1/6 documents)
7,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,examining the evidence,1,1,16.666667,6,0.85,I wrote down the evidence for and against my fear,16.7% (1/6 documents)
8,Emotional Response,The affective state or feeling of distress exp...,merged,emotional response,1,1,16.666667,6,1.00,I felt anxious about the meeting,16.7% (1/6 documents)
9,Graded Task Assignment,A behavioral strategy involving the breakdown ...,merged,graded task assignment,1,1,16.666667,6,1.00,I took one small step toward the task,16.7% (1/6 documents)


Each row shows one **discovered construct** and the **merged group** it belongs to.
One merged group can therefore appear on several rows.

| Column | In plain language |
|---|---|
| `merged_construct` | The name of the larger group, such as "Negative self-perception". |
| `definition` | What that group means. |
| `construct_found` | The name originally found in the documents, such as "self hate", after spelling cleanup. |
| `document_count` | How many different documents contained this discovered construct. Finding it three times in one document still counts as one document. |
| `instance_count` | How many nonempty supporting quote occurrences were found. Identical quotes from different documents still count separately. |
| `document_frequency` | How common it was: `1.7% (1/60 documents)` means one of the 60 successfully processed documents contained it. |
| `document_frequency_percent` | The same percentage as a number, useful for calculations. |
| `total_documents` | All successfully processed documents, including documents where nothing was found. Failed documents are excluded. |
| `prototypicality` | How well this discovered construct represents its merged group. A score near **1** means a strong, typical example; a score near **0** means a less typical member. This is **not** the confidence that a quote was detected correctly, or how often it appeared. |
| `quotes` | Supporting text. The display below shows up to three quotes per row, while the underlying results retain every unique quote text. |
| `status` | `merged`: put into a group. `standalone`: kept on its own. `unplaced`: left out by the model but kept by the package so it is not lost. |

An empty prototypicality score means no score is available; it does not mean zero.
Keep the raw `merge_table` for later steps. The formatted view is only for reading.
The `batch=True` option here uses sequential requests, not the asynchronous Batch API.


## 6. Review and apply manual grouping edits

The dictionary below is also saved as `merged_constructs_groups_<timestamp>.json`.
You can rename a key, move a constituent between lists, or omit a constituent to
remove it. Each known constituent may appear only once. This function makes no
LLM calls. For direct Python pasting, the following values contain only strings
and lists, so the printed JSON also works as a Python dictionary.


Settings for `apply_merge_edits()`:

| Setting | What it means |
|---|---|
| `table` | The original merge table, so counts and quotes can follow each construct into its edited group. |
| `grouping` | Your edited dictionary: each group name is followed by the list of discovered constructs that belong to it. |
| `definitions` | Optional new descriptions for groups you rename or change. Otherwise existing descriptions are kept where possible. |
| `total_documents` | Normally carried forward automatically. Editing the groups does not change how many documents were processed. |
| `output_dir` | Where to save the updated table and grouping dictionaries. |
| `save_format` | Choose CSV, JSON, or `"both"`. |

If you leave a discovered construct out of every list, it is removed. Listing the
same construct twice is an error. Counts and quotes are kept; this step does not
ask the LLM to redo discovery.


In [14]:
grouping = {
    name: group["construct_found"].tolist()
    for name, group in merge_table.groupby("merged_construct", sort=False)
}
print(json.dumps(grouping, indent=2, ensure_ascii=False))


{
  "Automatic Thought": [
    "automatic thought"
  ],
  "Avoidance Behavior": [
    "avoidance behavior",
    "overcoming behavioral avoidance"
  ],
  "Cognitive Distortion": [
    "cognitive distortion",
    "cognitive distortion all or nothing thinking",
    "cognitive distortion mind reading"
  ],
  "Cognitive Restructuring": [
    "cognitive restructuring",
    "examining the evidence"
  ],
  "Emotional Response": [
    "emotional response"
  ],
  "Graded Task Assignment": [
    "graded task assignment"
  ],
  "Intermediate Belief": [
    "intermediate belief conditional assumption"
  ]
}


In [15]:
# Replace this assignment with your edited dictionary, or edit the copy below.
edited_grouping = deepcopy(grouping)

# Examples, using names that actually occur in the printed dictionary:
# edited_grouping["New group name"] = edited_grouping.pop("Old group name")
# edited_grouping["Group name"].remove("constituent to remove")

edited_merged, edited_table = apply_merge_edits(
    table=merge_table,
    grouping=edited_grouping,
    # definitions={"New group name": "Your revised definition."},
    output_dir=edits_dir,
    save_format="both",
)
display(format_merged_constructs(edited_table, n_quotes=3, seed=seed))


Saved merge output to discovery_outputs/edited


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Automatic Thought,"An immediate, reflexive negative cognition tha...",merged,automatic thought,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
1,Avoidance Behavior,A behavioral response characterized by evading...,merged,avoidance behavior,1,1,16.666667,6,1.00,so I avoided opening the invitation.,16.7% (1/6 documents)
2,Avoidance Behavior,A behavioral response characterized by evading...,merged,overcoming behavioral avoidance,1,1,16.666667,6,0.75,instead of putting it off again.,16.7% (1/6 documents)
3,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
4,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion all or nothing thinking,1,1,16.666667,6,0.90,"If I make one mistake at work, everyone will t...",16.7% (1/6 documents)
5,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion mind reading,1,1,16.666667,6,0.90,everyone will think I am incompetent.,16.7% (1/6 documents)
6,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,cognitive restructuring,1,1,16.666667,6,1.00,which helped me consider another explanation.,16.7% (1/6 documents)
7,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,examining the evidence,1,1,16.666667,6,0.85,I wrote down the evidence for and against my fear,16.7% (1/6 documents)
8,Emotional Response,The affective state or feeling of distress exp...,merged,emotional response,1,1,16.666667,6,1.00,I felt anxious about the meeting,16.7% (1/6 documents)
9,Graded Task Assignment,A behavioral strategy involving the breakdown ...,merged,graded task assignment,1,1,16.666667,6,1.00,I took one small step toward the task,16.7% (1/6 documents)


Counts and quotes stay with their constituents. The denominator stays fixed.
Existing definitions are preserved where a source is identifiable; a new mixed
group may need an explicit definition. Moved members lose their old prototypicality
score because it described their previous group. A pure group rename retains
scores. Definitions can be outdated after membership changes; reevaluation below
can update names and definitions, but does not rescore prototypicality.


## 7. Filter the merge results

This is also local. Minimum thresholds are inclusive, and all supplied criteria
must pass. Removal names are matched using case/separator normalization. Unknown
removal names raise an error.

If `min_prototypicality` is supplied, missing scores are excluded—including
standalone/unplaced constructs and moved members. Leave it `None` to retain them.
Filtering never reduces the successful-document denominator.


In [16]:
remove_constructs = []       # Constituent names to remove.
remove_merged_constructs = []  # Whole merged groups to remove.
min_document_count = 1
min_instance_count = 1
min_prototypicality = None

filtered_merged, filtered_table = filter_merged_constructs(
    table=edited_table,
    remove_constructs=remove_constructs,
    remove_merged_constructs=remove_merged_constructs,
    min_document_count=min_document_count,
    min_instance_count=min_instance_count,
    min_prototypicality=min_prototypicality,
    output_dir=filtered_dir,
    save_format="both",
)
display(format_merged_constructs(filtered_table, n_quotes=3, seed=seed))
print("Constituents remaining:", len(filtered_table))
print("Removed members recorded:", len(filtered_table.attrs.get("removed_constituents", [])))

if filtered_table.empty:
    raise RuntimeError("All constituents were removed. Relax the filters or revise the grouping before building a codebook.")


Saved merge output to discovery_outputs/filtered


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Automatic Thought,"An immediate, reflexive negative cognition tha...",merged,automatic thought,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
1,Avoidance Behavior,A behavioral response characterized by evading...,merged,avoidance behavior,1,1,16.666667,6,1.00,so I avoided opening the invitation.,16.7% (1/6 documents)
2,Avoidance Behavior,A behavioral response characterized by evading...,merged,overcoming behavioral avoidance,1,1,16.666667,6,0.75,instead of putting it off again.,16.7% (1/6 documents)
3,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
4,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion all or nothing thinking,1,1,16.666667,6,0.90,"If I make one mistake at work, everyone will t...",16.7% (1/6 documents)
5,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion mind reading,1,1,16.666667,6,0.90,everyone will think I am incompetent.,16.7% (1/6 documents)
6,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,cognitive restructuring,1,1,16.666667,6,1.00,which helped me consider another explanation.,16.7% (1/6 documents)
7,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,examining the evidence,1,1,16.666667,6,0.85,I wrote down the evidence for and against my fear,16.7% (1/6 documents)
8,Emotional Response,The affective state or feeling of distress exp...,merged,emotional response,1,1,16.666667,6,1.00,I felt anxious about the meeting,16.7% (1/6 documents)
9,Graded Task Assignment,A behavioral strategy involving the breakdown ...,merged,graded task assignment,1,1,16.666667,6,1.00,I took one small step toward the task,16.7% (1/6 documents)


Constituents remaining: 11
Removed members recorded: 0


## 8. Optionally reevaluate merged names and definitions

Set `run_reevaluation=True` in the options cell to enable this extra LLM call.
The model receives the previous groups, current groups, and removed constituents.
It may revise names and definitions only; it cannot move or restore constituents.
Quotes, counts, the denominator, and existing/missing scores stay unchanged.

The template is available as `REEVALUATE_PROMPT`. A custom template uses
`{framework}`, `{previous}`, `{removed}`, and `{current}` and must keep the response
schema. The full prompt is checked against `max_prompt_chars` before sending.


In [17]:
# Print this to inspect the reevaluation instructions without an API call:
# print(REEVALUATE_PROMPT)

if run_reevaluation:
    final_merged, final_table = reevaluate_merged_constructs(
        table=filtered_table,
        config=config,
        framework=framework,
        reevaluation_prompt=REEVALUATE_PROMPT,
        max_prompt_chars=60000,
        output_dir=reevaluation_dir,
        save_format="both",
    )
else:
    final_merged, final_table = filtered_merged, filtered_table
    print("Reevaluation skipped; using the locally reviewed and filtered result.")

display(format_merged_constructs(final_table, n_quotes=3, seed=seed))


Reevaluation skipped; using the locally reviewed and filtered result.


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Automatic Thought,"An immediate, reflexive negative cognition tha...",merged,automatic thought,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
1,Avoidance Behavior,A behavioral response characterized by evading...,merged,avoidance behavior,1,1,16.666667,6,1.00,so I avoided opening the invitation.,16.7% (1/6 documents)
2,Avoidance Behavior,A behavioral response characterized by evading...,merged,overcoming behavioral avoidance,1,1,16.666667,6,0.75,instead of putting it off again.,16.7% (1/6 documents)
3,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion,1,1,16.666667,6,1.00,I assume my friend is angry whenever she takes...,16.7% (1/6 documents)
4,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion all or nothing thinking,1,1,16.666667,6,0.90,"If I make one mistake at work, everyone will t...",16.7% (1/6 documents)
5,Cognitive Distortion,A pattern of biased or irrational thinking tha...,merged,cognitive distortion mind reading,1,1,16.666667,6,0.90,everyone will think I am incompetent.,16.7% (1/6 documents)
6,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,cognitive restructuring,1,1,16.666667,6,1.00,which helped me consider another explanation.,16.7% (1/6 documents)
7,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",merged,examining the evidence,1,1,16.666667,6,0.85,I wrote down the evidence for and against my fear,16.7% (1/6 documents)
8,Emotional Response,The affective state or feeling of distress exp...,merged,emotional response,1,1,16.666667,6,1.00,I felt anxious about the meeting,16.7% (1/6 documents)
9,Graded Task Assignment,A behavioral strategy involving the breakdown ...,merged,graded task assignment,1,1,16.666667,6,1.00,I took one small step toward the task,16.7% (1/6 documents)


## 9. Save all quotes and a construct summary

We will save two tables without making any LLM calls:

1. **All quotes:** one row for each quote occurrence, with the source file,
   document ID, original discovered name, cleaned name, and final merged group.
   Identical quote text from different documents remains on separate rows.
2. **Construct summary:** one row per merged group, with the columns below.
   It contains no additional Overall row.

The quote table includes every original quote occurrence, even for constituents
removed during review. Its `retained` column tells you whether the constituent is
still included; removed constituents have no final merged group. The summary
contains only the groups you kept after editing and filtering.

| Summary column | What it tells you |
|---|---|
| Construct | The final merged group name. |
| Definition | What that group means. |
| Construct found % (N) | How many different discovered constructs belong to this group, as a share of all originally discovered constructs after name cleanup. |
| Document count % (N) | How many different documents contain this group, as a share of all successfully processed documents. A document with several members of the same group counts only once. |
| Instance Count % (N) | How many supporting quote occurrences belong to this group, as a share of all originally discovered quote occurrences. |
| Prototypicality mean (SD) | The average of its available constituent scores. SD describes how spread out those scores are: smaller SD means more similar scores. A single score has no SD; missing scores are ignored rather than treated as zero. |
| Examples | All unique quote texts for this group. This is not limited to three. |
| Constituents found | The complete list of discovered constructs kept in this group. |

Percentages show both counts: `50.0% (2/4)` means two out of four. The original
totals stay fixed after filtering. Percentages across different groups can overlap
for documents because the same document may contain more than one group.

`build_discovery_tables()` takes the **original `discoveries`**, the raw final
`final_table`, and the same `fuzzy_threshold` used for merging. `output_dir` chooses
the folder; `save_format="both"` preserves spreadsheet CSVs and list-friendly JSON.
The saved context JSON records the original totals used for the percentages.


In [18]:
all_quotes, construct_summary = build_discovery_tables(
    discoveries=discoveries,
    merged_table=final_table,
    fuzzy_threshold=fuzzy_threshold,
    output_dir=tables_dir,
    save_format="both",
)

# Both tables are saved in full; pandas may shorten the on-screen preview.
with pd.option_context("display.max_colwidth", None):
    display(all_quotes)
    display(construct_summary)
print("Final tables saved to:", tables_dir.resolve())


Saved all original quote occurrences and final construct summary to discovery_outputs/final_tables


,filename,doc_id,merged_construct,construct_found,original_construct,description,quote,retained
0,discovery_demo.csv,demo_1,Cognitive Distortion,cognitive distortion mind reading,Cognitive Distortion: Mind Reading,The individual assumes they know the thoughts and judgments of others without any objective evidence.,everyone will think I am incompetent.,True
1,discovery_demo.csv,demo_1,Intermediate Belief,intermediate belief conditional assumption,Intermediate Belief: Conditional Assumption,"The individual operates under a rigid, dysfunctional if-then rule that links a specific behavior to a global negative judgment.","If I make one mistake at work, everyone will think I am incompetent.",True
2,discovery_demo.csv,demo_1,Cognitive Distortion,cognitive distortion all or nothing thinking,Cognitive Distortion: All-or-Nothing Thinking,The individual views a single flaw or error as indicative of complete failure or total lack of ability.,"If I make one mistake at work, everyone will think I am incompetent.",True
3,discovery_demo.csv,demo_2,Cognitive Distortion,cognitive distortion,Cognitive distortion,"The individual exhibits a pattern of biased thinking, specifically mind reading or jumping to conclusions, by assuming another person's negative emotional state without direct evidence.",I assume my friend is angry whenever she takes a while to reply.,True
4,discovery_demo.csv,demo_2,Automatic Thought,automatic thought,Automatic thought,"The text illustrates an immediate, reflexive negative cognition that arises in response to a specific situational trigger.",I assume my friend is angry whenever she takes a while to reply.,True
5,discovery_demo.csv,demo_3,Avoidance Behavior,avoidance behavior,Avoidance Behavior,"A behavioral response characterized by evading a distressing stimulus or situation, which cognitive behavioral therapy identifies as a mechanism that maintains anxiety.",so I avoided opening the invitation.,True
6,discovery_demo.csv,demo_3,Emotional Response,emotional response,Emotional Response,The affective state or feeling of distress experienced by the individual in anticipation of a specific event.,I felt anxious about the meeting,True
7,discovery_demo.csv,demo_4,Cognitive Restructuring,examining the evidence,Examining the evidence,The process of objectively evaluating the factual support for and against a specific anxious thought or belief.,I wrote down the evidence for and against my fear,True
8,discovery_demo.csv,demo_4,Cognitive Restructuring,cognitive restructuring,Cognitive restructuring,"The process of developing new, more balanced or realistic thoughts and interpretations to replace maladaptive ones.",which helped me consider another explanation.,True
9,discovery_demo.csv,demo_5,Graded Task Assignment,graded task assignment,Graded task assignment,"The cognitive behavioral strategy of breaking down a larger, potentially overwhelming activity into smaller, manageable actions to facilitate task initiation.",I took one small step toward the task,True


,Construct,Definition,Construct found % (N),Document count % (N),Instance Count % (N),Prototypicality mean (SD),Examples,Constituents found
0,Automatic Thought,"An immediate, reflexive negative cognition that arises spontaneously in response to a specific situational trigger.",9.1% (1/11),16.7% (1/6),9.1% (1/11),1.00 (N/A),[I assume my friend is angry whenever she takes a while to reply.],[automatic thought]
1,Avoidance Behavior,"A behavioral response characterized by evading distressing stimuli or situations, which maintains anxiety and is targeted for modification in therapy.",18.2% (2/11),33.3% (2/6),18.2% (2/11),0.88 (0.18),"[so I avoided opening the invitation., instead of putting it off again.]","[avoidance behavior, overcoming behavioral avoidance]"
2,Cognitive Distortion,"A pattern of biased or irrational thinking that negatively influences an individual's perception of reality, such as mind reading or all-or-nothing thinking.",27.3% (3/11),33.3% (2/6),27.3% (3/11),0.93 (0.06),"[everyone will think I am incompetent., If I make one mistake at work, everyone will think I am incompetent., I assume my friend is angry whenever she takes a while to reply.]","[cognitive distortion, cognitive distortion all or nothing thinking, cognitive distortion mind reading]"
3,Cognitive Restructuring,"The therapeutic process of identifying, evaluating the evidence for, and modifying maladaptive thoughts to develop more balanced and realistic perspectives.",18.2% (2/11),16.7% (1/6),18.2% (2/11),0.93 (0.11),"[I wrote down the evidence for and against my fear, which helped me consider another explanation.]","[cognitive restructuring, examining the evidence]"
4,Emotional Response,The affective state or feeling of distress experienced by an individual in anticipation of or in response to a specific event.,9.1% (1/11),16.7% (1/6),9.1% (1/11),1.00 (N/A),[I felt anxious about the meeting],[emotional response]
5,Graded Task Assignment,"A behavioral strategy involving the breakdown of overwhelming activities into smaller, manageable steps to facilitate task initiation.",9.1% (1/11),16.7% (1/6),9.1% (1/11),1.00 (N/A),[I took one small step toward the task],[graded task assignment]
6,Intermediate Belief,"A rigid, dysfunctional rule, attitude, or conditional assumption that links specific behaviors to global negative judgments.",9.1% (1/11),16.7% (1/6),9.1% (1/11),1.00 (N/A),"[If I make one mistake at work, everyone will think I am incompetent.]",[intermediate belief conditional assumption]


Final tables saved to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/final_tables


## 10. Build, review, and save a codebook

Codebook creation is local. Each merged construct becomes a codebook entry with
its definition, sampled example quotes, and an initially empty counterexample
list. Quotes are deduplicated and sampled from the pooled quotes for each group;
they are not balanced across constituents.

Review definitions and examples before coding. Keep list columns as real Python
lists when editing; display strings can split quotes containing semicolons.


Settings for `build_codebook()`:

| Setting | What it means |
|---|---|
| `merged` | The final groups you decided to keep, including any revised names and definitions. |
| `name` | A label for this codebook, so you can recognize it later. |
| `version` | Your version number; increase it when making a new reviewed version. |
| `n_examples` | The largest number of example quotes to put into each codebook entry. It does not remove quotes from the discovery results or the tables saved in Step 9. |
| `seed` | Makes the choice of example quotes repeatable for the same input and number. |
| `output_path` | Optional filename for saving immediately. Here we review the codebook first and save it in the following cell. Saving adds a timestamp. |

The codebook starts with an empty counterexample list for each construct. You can
add examples of what should **not** count during your review.


In [19]:
codebook = build_codebook(
    merged=final_merged,
    name="Discovery tutorial codebook",
    version=1,
    n_examples=n_codebook_examples,
    seed=seed,
)
codebook_frame = codebook_to_frame(codebook)
display(codebook_frame)


,construct,definition,examples,counter_examples
0,Automatic Thought,"An immediate, reflexive negative cognition tha...",[I assume my friend is angry whenever she take...,[]
1,Avoidance Behavior,A behavioral response characterized by evading...,"[so I avoided opening the invitation., instead...",[]
2,Cognitive Distortion,A pattern of biased or irrational thinking tha...,[I assume my friend is angry whenever she take...,[]
3,Cognitive Restructuring,"The therapeutic process of identifying, evalua...",[which helped me consider another explanation....,[]
4,Emotional Response,The affective state or feeling of distress exp...,[I felt anxious about the meeting],[]
5,Graded Task Assignment,A behavioral strategy involving the breakdown ...,[I took one small step toward the task],[]
6,Intermediate Belief,"A rigid, dysfunctional rule, attitude, or cond...","[If I make one mistake at work, everyone will ...",[]


In [20]:
# Make any manual edits to codebook_frame before running this cell.
# Example:
# codebook_frame.at[0, "definition"] = "A clearer definition."
# codebook_frame.at[0, "counter_examples"] = ["A quote that should not count."]

if codebook_frame["definition"].fillna("").str.strip().eq("").any():
    raise ValueError("Fill in missing codebook definitions before saving.")

reviewed_codebook = frame_to_codebook(
    codebook_frame,
    metadata=codebook["metadata"],
)
saved_codebook_path = save_codebook(
    reviewed_codebook,
    codebook_dir / "discovered_codebook.json",
)
print("Use this codebook path in the coding pipeline:", saved_codebook_path.resolve())


Saved codebook to discovery_outputs/codebooks/discovered_codebook_20261110_155200_336416.json
Use this codebook path in the coding pipeline: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/codebooks/discovered_codebook_20261110_155200_336416.json
